# Sistemas aparentemente no relacionados (SUR)

**Unidad 2.c del temario** · **Notas de Clase: capítulo 4, §4.4**

Dos ecuaciones pueden no compartir ningún regresor y aun así **no ser independientes**:
si sus perturbaciones están correlacionadas, estimarlas por separado desperdicia
información. Ésa es la idea de Zellner (1962), y las ecuaciones se llaman *aparentemente*
no relacionadas porque la relación no está en la parte sistemática sino en el error.

El caso canónico —el del artículo original— son las funciones de inversión de Grunfeld
(1958), y es el que replicamos aquí.

## El modelo

Para cada empresa $g = 1, \dots, G$:

$$I_{gt} = \beta_{g0} + \beta_{g1} V_{gt} + \beta_{g2} K_{gt} + \varepsilon_{gt}$$

- $I$: inversión bruta
- $V$: valor de mercado de la empresa al inicio del año
- $K$: acervo de capital instalado

El supuesto que une el sistema es la **correlación contemporánea**:

$$\mathbb{E}[\varepsilon_{gt}\varepsilon_{ht}] = \sigma_{gh}, \qquad
\mathbb{E}[\varepsilon_{gt}\varepsilon_{hs}] = 0 \text{ si } t \ne s$$

Choques macroeconómicos comunes —una recesión, un cambio en la tasa de interés— afectan a
todas las empresas el mismo año, y eso es exactamente lo que $\sigma_{gh} \ne 0$ recoge.

In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy.stats import chi2

# Los datos de Grunfeld (1958) vienen incluidos en statsmodels: no hay que descargarlos.
from statsmodels.datasets import grunfeld

datos = grunfeld.load_pandas().data
datos["year"] = datos["year"].astype(int)

print(f"Observaciones: {len(datos)}   Empresas: {datos['firm'].nunique()}")
print(f"Periodo: {datos['year'].min()}-{datos['year'].max()}\n")
print(datos["firm"].unique())

Observaciones: 220   Empresas: 11
Periodo: 1935-1954

['General Motors' 'US Steel' 'General Electric' 'Chrysler'
 'Atlantic Refining' 'IBM' 'Union Oil' 'Westinghouse' 'Goodyear'
 'Diamond Match' 'American Steel']


Zellner (1962) trabaja con **dos** empresas —General Motors y Westinghouse—, que es el
caso que se reproduce en los libros de texto. Elegirlas no es arbitrario: son de tamaños
muy distintos, lo que hace visible el problema de escala, y es el ejemplo con el que se
puede contrastar contra números publicados.

In [2]:
EMPRESAS = ["General Motors", "Westinghouse"]
T = 20

sistema = {}
for empresa in EMPRESAS:
    s = datos[datos["firm"] == empresa].sort_values("year")
    sistema[empresa] = {
        "y": s["invest"].values,
        "X": sm.add_constant(s[["value", "capital"]]).values,
    }

print(f"{'':16s} {'inversión media':>16s} {'valor medio':>14s}")
for empresa in EMPRESAS:
    s = datos[datos["firm"] == empresa]
    print(f"{empresa:16s} {s['invest'].mean():16.2f} {s['value'].mean():14.2f}")
print("\nGM es un orden de magnitud mayor que Westinghouse.")

                  inversión media    valor medio
General Motors             608.02        4333.84
Westinghouse                42.89         670.91

GM es un orden de magnitud mayor que Westinghouse.


## Paso 1 — MCO ecuación por ecuación

Es el punto de partida y también el estimador contra el cual se compara. Bajo el supuesto
de errores correlacionados, MCO por ecuación sigue siendo **insesgado y consistente**,
pero **no eficiente**.

In [3]:
mco = {}
residuales = []
for empresa in EMPRESAS:
    ajuste = sm.OLS(sistema[empresa]["y"], sistema[empresa]["X"]).fit()
    mco[empresa] = ajuste
    residuales.append(ajuste.resid)
    print(f"{empresa}")
    print(f"  constante {ajuste.params[0]:10.4f}  (ee {ajuste.bse[0]:.4f})")
    print(f"  value     {ajuste.params[1]:10.4f}  (ee {ajuste.bse[1]:.4f})")
    print(f"  capital   {ajuste.params[2]:10.4f}  (ee {ajuste.bse[2]:.4f})")
    print(f"  R^2 = {ajuste.rsquared:.4f}\n")

General Motors
  constante  -149.7825  (ee 105.8421)
  value         0.1193  (ee 0.0258)
  capital       0.3714  (ee 0.0371)
  R^2 = 0.9214

Westinghouse
  constante    -0.5094  (ee 8.0153)
  value         0.0529  (ee 0.0157)
  capital       0.0924  (ee 0.0561)
  R^2 = 0.7444



### Verificación contra el resultado publicado

La ecuación de General Motors es la que aparece en Zellner (1962) y se reproduce en
Greene (2012):

In [4]:
PUBLICADO_GM = {"const": -149.78, "value": 0.1193, "capital": 0.3714}

gm = mco["General Motors"]
print(f"{'':10s} {'estimado':>12s} {'publicado':>12s} {'dif.':>10s}")
for i, (nombre, valor) in enumerate(PUBLICADO_GM.items()):
    print(f"{nombre:10s} {gm.params[i]:12.4f} {valor:12.4f} {gm.params[i] - valor:10.4f}")

print("\nLa ecuación de GM replica los valores publicados.")

               estimado    publicado       dif.
const         -149.7825    -149.7800    -0.0025
value            0.1193       0.1193    -0.0000
capital          0.3714       0.3714     0.0000

La ecuación de GM replica los valores publicados.


> **Una advertencia sobre estos datos, que vale como lección de método.** Circulan varias
> versiones incompatibles de los datos de Grunfeld: difieren en algunas empresas por
> errores de transcripción arrastrados durante décadas. Kleiber y Zeileis (2010)
> documentaron el problema comparando las copias usadas en distintos libros de texto.
>
> Que la ecuación de GM replique exactamente **no garantiza** que las demás lo hagan.
> El ejercicio 5 del final pide contrastar la de Westinghouse contra la tabla de algún
> libro de texto y averiguar si coincide. **Verificar una ecuación no es verificar el
> archivo.**

## Paso 2 — La matriz de covarianzas contemporánea

Se estima con los residuales de MCO:

$$\hat{\sigma}_{gh} = \frac{1}{T}\sum_{t} \hat{\varepsilon}_{gt}\hat{\varepsilon}_{ht}$$

In [5]:
E = np.column_stack(residuales)
Sigma = E.T @ E / T
correlacion = Sigma[0, 1] / np.sqrt(Sigma[0, 0] * Sigma[1, 1])

print("Sigma estimada:")
print(pd.DataFrame(Sigma, index=EMPRESAS, columns=EMPRESAS).round(2).to_string())
print(f"\nCorrelación de los residuales: {correlacion:.4f}")

Sigma estimada:
                General Motors  Westinghouse
General Motors         7160.29        126.18
Westinghouse            126.18         88.66

Correlación de los residuales: 0.1584


### ¿Hace falta SUR? La prueba de diagonalidad

Si $\Sigma$ fuera diagonal, SUR y MCO coincidirían y no habría nada que ganar. La prueba
de multiplicadores de Lagrange de Breusch-Pagan contrasta $H_0: \sigma_{gh} = 0$ para todo
$g \ne h$ (capítulo 4, §4.4 de las notas). Con dos ecuaciones:

$$LM = T \, r_{12}^2 \;\sim\; \chi^2_{1}$$

In [6]:
LM = T * correlacion ** 2
p_valor = 1 - chi2.cdf(LM, 1)

print(f"LM = {LM:.4f}   p = {p_valor:.4f}")
print()
if p_valor > 0.05:
    print("No se rechaza la diagonalidad al 5 %. La correlación entre los")
    print("errores de GM y Westinghouse es baja (0.16), de modo que las")
    print("ganancias de eficiencia de SUR deben ser PEQUEÑAS.")
    print()
    print("Conviene anticiparlo antes de estimar: así el resultado del paso 3")
    print("confirma o desmiente una predicción, en vez de sorprendernos.")

LM = 0.5016   p = 0.4788

No se rechaza la diagonalidad al 5 %. La correlación entre los
errores de GM y Westinghouse es baja (0.16), de modo que las
ganancias de eficiencia de SUR deben ser PEQUEÑAS.

Conviene anticiparlo antes de estimar: así el resultado del paso 3
confirma o desmiente una predicción, en vez de sorprendernos.


## Paso 3 — El estimador SUR

SUR es MCG sobre el sistema apilado, con matriz de covarianzas
$\Omega = \Sigma \otimes I_T$:

$$\hat{\boldsymbol\beta}_{SUR} =
\left[\mathbf{X}'(\Sigma^{-1} \otimes I_T)\mathbf{X}\right]^{-1}
\mathbf{X}'(\Sigma^{-1} \otimes I_T)\mathbf{y}$$

donde $\mathbf{X}$ es **diagonal por bloques**: cada ecuación tiene sus propios
regresores. Lo programamos directamente para que se vea la estructura.

In [7]:
K = 3                      # regresores por ecuación (constante, value, capital)
G = len(EMPRESAS)

y_apilada = np.concatenate([sistema[e]["y"] for e in EMPRESAS])

X_bloques = np.zeros((G * T, G * K))
for g, empresa in enumerate(EMPRESAS):
    X_bloques[g * T:(g + 1) * T, g * K:(g + 1) * K] = sistema[empresa]["X"]

Omega_inversa = np.kron(np.linalg.inv(Sigma), np.eye(T))

XtOX = X_bloques.T @ Omega_inversa @ X_bloques
beta_sur = np.linalg.solve(XtOX, X_bloques.T @ Omega_inversa @ y_apilada)
ee_sur = np.sqrt(np.diag(np.linalg.inv(XtOX)))

etiquetas = [f"{e.split()[0]}_{v}" for e in EMPRESAS for v in ["const", "value", "capital"]]
beta_mco = np.concatenate([mco[e].params for e in EMPRESAS])
ee_mco = np.concatenate([mco[e].bse for e in EMPRESAS])

resultados = pd.DataFrame(
    {"MCO": beta_mco, "ee_MCO": ee_mco, "SUR": beta_sur, "ee_SUR": ee_sur},
    index=etiquetas,
)
resultados["ganancia_%"] = 100 * (1 - resultados["ee_SUR"] / resultados["ee_MCO"])
resultados.round(4)

,MCO,ee_MCO,SUR,ee_SUR,ganancia_%
General_const,-149.7825,105.8421,-144.4919,97.1407,8.2211
General_value,0.1193,0.0258,0.1181,0.0237,8.2845
General_capital,0.3714,0.0371,0.3713,0.0341,8.0833
Westinghouse_const,-0.5094,8.0153,0.0670,7.3669,8.0890
Westinghouse_value,0.0529,0.0157,0.0522,0.0144,8.2550
Westinghouse_capital,0.0924,0.0561,0.0914,0.0515,8.2580


## Qué muestra el cuadro

**Los coeficientes casi no se mueven**, como habíamos anticipado a partir de la prueba de
diagonalidad. La ganancia está en los **errores estándar**, que caen alrededor de un **8 %** —de forma
notablemente uniforme en los seis coeficientes— sin cambiar los datos ni el modelo. Es
eficiencia pura, obtenida por usar la información de que los errores están correlacionados.

Como el error estándar decrece con $\sqrt{N}$, reducirlo un 8 % equivale en precisión a
haber recolectado cerca de un **19 % más de observaciones**. Con una correlación de apenas
0.16.

> **La regla que conviene retener.** Las ganancias de SUR son mayores cuando
> (i) la correlación entre errores es **alta** y (ii) los regresores de las ecuaciones son
> **distintos**. En el extremo opuesto —regresores idénticos en todas las ecuaciones—
> SUR **coincide exactamente** con MCO ecuación por ecuación. Ese teorema de equivalencia,
> con sus dos demostraciones, está en el capítulo 4 de las notas.

In [8]:
# Comprobación del teorema de equivalencia: si las dos ecuaciones tienen
# EXACTAMENTE los mismos regresores, SUR debe reproducir a MCO.
X_comun = sistema["General Motors"]["X"]

X_igual = np.zeros((G * T, G * K))
for g in range(G):
    X_igual[g * T:(g + 1) * T, g * K:(g + 1) * K] = X_comun

XtOX_i = X_igual.T @ Omega_inversa @ X_igual
beta_sur_igual = np.linalg.solve(XtOX_i, X_igual.T @ Omega_inversa @ y_apilada)

beta_mco_igual = np.concatenate(
    [sm.OLS(sistema[e]["y"], X_comun).fit().params for e in EMPRESAS]
)

print(f"Diferencia máxima |SUR - MCO| con regresores idénticos: "
      f"{np.abs(beta_sur_igual - beta_mco_igual).max():.2e}")
print("\nEl teorema de equivalencia se cumple numéricamente.")

Diferencia máxima |SUR - MCO| con regresores idénticos: 4.15e-12

El teorema de equivalencia se cumple numéricamente.


Esta comprobación es un ejemplo del criterio de verificación de la
[unidad 4](../Clase_15_CodigoAsistidoPorIA/): en lugar de confiar en que el código está
bien, se le exige **cumplir una identidad que la teoría predice**. Si el estimador SUR
estuviera mal programado, esta prueba lo delataría.

---

## Ejercicios

1. **Todas las empresas.** Extiende el sistema a las 11 empresas del archivo. ¿Cómo
   cambia la prueba de diagonalidad? ¿Y las ganancias de eficiencia? *Cuidado:* con
   $G = 11$ y $T = 20$, $\Sigma$ tiene 66 parámetros distintos estimados con 20
   observaciones cada uno. ¿Es confiable el $\hat\Sigma$ resultante?
2. **SUR iterado.** El estimador que calculamos es SUR factible en **una etapa**: usa
   $\hat\Sigma$ de los residuales de MCO. Itera —recalcula $\hat\Sigma$ con los residuales
   de SUR, reestima, y repite hasta convergencia— y compara. Bajo normalidad, el límite
   es el estimador de máxima verosimilitud (capítulo 6).
3. **Restricciones entre ecuaciones.** Contrasta $H_0$: el coeficiente de `value` es el
   mismo en ambas empresas. Esta prueba **no se puede hacer** con MCO ecuación por
   ecuación, y es una de las razones principales para estimar el sistema conjuntamente.
4. **Relación con datos panel.** Compara el SUR de este cuaderno con los estimadores de
   [`Clase_04_DatosPanel`](../Clase_04_DatosPanel/). ¿Qué supone cada uno sobre la
   heterogeneidad entre empresas? *Pista:* el capítulo 5 muestra que efectos aleatorios es
   MCG sobre un sistema con coeficientes restringidos a ser iguales.
5. **Verificar el archivo, no sólo una ecuación.** Contrasta los coeficientes de
   Westinghouse contra los de algún libro de texto que reproduzca este ejemplo. ¿Coinciden
   como los de GM? Si no, revisa Kleiber y Zeileis (2010) y determina qué versión de los
   datos estás usando. **Éste es el ejercicio más importante de la lista.**

---

## Referencias

- **Zellner, A. (1962).** «An efficient method of estimating seemingly unrelated
  regressions and tests for aggregation bias», *JASA* 57(298): 348-368.
- **Grunfeld, Y. (1958).** *The Determinants of Corporate Investment.* Tesis doctoral,
  Universidad de Chicago.
- **Kleiber, C. y A. Zeileis (2010).** «The Grunfeld data at 50», *German Economic Review*
  11(4): 404-417.
- **Breusch, T. y A. Pagan (1980).** «The Lagrange multiplier test and its applications to
  model specification in econometrics», *Review of Economic Studies* 47(1): 239-253.

---
Parte del curso de **Econometría I**, Facultad de Ciencias, UNAM.
Teoría en el **capítulo 4, §4.4** de las Notas de Clase.